# 07 · Paper consistency checks
Pre-submission checks on `paper/`: every `\cite` key resolves in the bibliography, every result macro used in the text is
generated by the analysis, the figure exists, and phrases flagged by the novelty review (e.g. unsupported "first" claims) are
absent.

In [1]:
import sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT / "harness"), str(ROOT / "benchmark")]
import pandas as pd
from IPython.display import Markdown, Image, display
pd.set_option("display.max_colwidth", 120)
print("repository root:", ROOT)

repository root: C:\IEEE-RESEARCH-WORK


In [2]:
import re, hashlib
tex ="\n".join(p.read_text(encoding="utf-8") for p in sorted((ROOT / "paper").glob("*.tex")))
body = re.sub(r"(?<!\\)%.*", "", tex)
bibkeys = set(re.findall(r"@\w+\s*\{\s*([^,\s]+)\s*,", (ROOT / "paper/references.bib").read_text(encoding="utf-8")))
cited = {k.strip() for g in re.findall(r"\\cite\w*\{([^}]*)\}", body) for k in g.split(",")}
missing = sorted(cited - bibkeys)
print(f"cited keys: {len(cited)}; missing from references.bib: {missing}")
# the paper's macro files must be byte-identical to the analysis output, and name the intended runs and model
gen = {"results_macros.tex": (ROOT / "pilot/analysis/results_macros.tex", "qwen3:8b"),
       "results_macros_llama.tex": (ROOT / "pilot/analysis_llama/results_macros.tex", "llama3.1-tools:8b")}
defined = set()
for name, (src, model) in gen.items():
    f = ROOT / "paper/generated" / name
    if not f.exists():
        continue
    text = f.read_text(encoding="utf-8")
    assert text == src.read_text(encoding="utf-8"), f"paper/generated/{name} is stale: re-run notebook 04"
    assert model in text.splitlines()[0], f"{name} was generated from another model: {text.splitlines()[0]}"
    m = re.search(r"from (\S+) \(sha256 ([0-9a-f]{64})\)", text.splitlines()[0])   # provenance: the run log's raw bytes
    assert m and hashlib.sha256((ROOT / "pilot" / m[1]).read_bytes()).hexdigest() == m[2], f"{name}: its run log changed after the macros were generated"
    print(text.splitlines()[0])
    defined |= set(re.findall(r"\\newcommand\{\\(\w+)\}", text))
used = set(re.findall(r"\\((?:Res|Con|Ideal|HFive|Gate|Abl|Pow|Uat|Post|Llama|N)[A-Z]\w*)", body))
print(f"result macros used: {len(used)}; undefined: {sorted(used - defined)}")
banned = [r"\bthe first\b", r"\bfirst (?:study|benchmark|work)\b", r"to the best of our knowledge", r"\bguarantee", r"\bunprecedented\b",
          r"\bproduction-grade\b", r"\bhallucination-free\b", r"\bholistic\b"]
hits = {b: len(re.findall(b, body, re.I)) for b in banned if re.search(b, body, re.I)}
print("banned phrases found:", hits or "none")
labels = set(re.findall(r"\\label\{((?:fig|tab):[^}]*)\}", body))
unref = labels - set(re.findall(r"\\ref\{([^}]*)\}", body))
print("figure/table labels never referenced:", sorted(unref) or "none")
left = re.findall(r"\\pilot\{|\[(?:RESULT|Interpretation|Restate|Dominant|Department|City|email|First name|repository URL)", body)
print(f"draft markers still present (must be 0 before submission): {len(left)}")
assert not missing and not (used - defined) and not hits and not unref

cited keys: 36; missing from references.bib: []
% generated by harness/analyze.py from runs_dev_qwen3-8b.jsonl (sha256 5e8844bb83d32cfcbad9d0945ca50c827c06467abd62e32917ccc47c7ff3f590), 600 runs, model(s) qwen3:8b; do not edit by hand
% generated by harness/analyze.py from runs_dev_llama3.1-8b.jsonl (sha256 4e4f5ac686f4ad555988f355db81695cd0a230c1c1475e1571d938d7e5f23809), 180 runs, model(s) llama3.1-tools:8b; do not edit by hand
result macros used: 72; undefined: []
banned phrases found: none
figure/table labels never referenced: none
draft markers still present (must be 0 before submission): 20


In [3]:
import shutil, subprocess
tectonic = shutil.which("tectonic") or next((str(p) for p in (Path.home() / "tools/tectonic" / n for n in ("tectonic.exe", "tectonic"))
                                             if p.exists()), None)
if tectonic:  # builds with the conference's IEEEtran.cls / IEEEtran.bst copied into paper/
    r = subprocess.run([tectonic, "-X", "compile", "main.tex", "--outdir", "build"], cwd=ROOT / "paper", capture_output=True, text=True)
    print("tectonic:", "ok" if r.returncode == 0 else r.stderr[-800:])
else:
    print("Tectonic not found: checking the committed PDF, not a fresh build")
pdf = next((p for p in (ROOT / "paper/build/main.pdf", ROOT / "paper/main.pdf") if p.exists()), None)
if pdf:
    from pypdf import PdfReader          # counts pages correctly even with compressed object streams
    rd = PdfReader(pdf)
    fonts = {str(v.get_object().get("/Subtype")) for p in rd.pages for v in ((p.get("/Resources") or {}).get("/Font") or {}).values()}
    box = rd.pages[0].mediabox
    print(f"{pdf.relative_to(ROOT)}: {len(rd.pages)} pages, {pdf.stat().st_size / 1e6:.2f} MB, page {float(box.width) / 72:.1f} x "
          f"{float(box.height) / 72:.1f} in, font types {sorted(fonts)} (limits: 6 pages, 2 MB, no Type 3)")
    assert len(rd.pages) <= 6 and pdf.stat().st_size < 2e6 and "/Type3" not in fonts
else:
    print("PDF not built: install Tectonic or compile paper/main.tex on Overleaf (see paper/README.md)")

tectonic: ok
paper\build\main.pdf: 6 pages, 0.11 MB, page 8.5 x 11.0 in, font types ['/Type1'] (limits: 6 pages, 2 MB, no Type 3)
